**Что добавлено и исправлено в этой версии (ветка `ai-beatmaker`)**
> - Добавлено обучение autoregressive Transformer на последовательностях MIDI-событий с разбиением train/validation по композициям.
> - Добавлены сохранение checkpoint во время эпохи и после каждой эпохи, отдельный лучший checkpoint и возобновление через `--resume`.
> - Добавлены загрузчик PDMX с проверкой контрольных сумм, фильтрацией Public Domain/CC0 и продолжением загрузки/распаковки.
> - Добавлена сборка JSONL-датасета с проверкой лицензий, SHA-256-дедупликацией и восстановлением после прерывания.
> - Добавлены MIDI-конвертер, синтетический генератор, MIDI-сэмплер и экспорт результата в JSON + MIDI.
> - Важно: это генерация **символических MIDI-событий**, не генерация готового аудио или вокала. PyTorch-модель пока не подключена к Android-приложению.
> - Цель 200 000 — запрашиваемый максимум (10 000 синтетических + до 190 000 внешних). Наличие такого количества подходящих MIDI не гарантируется.
>
> Изменения внесены в GitHub-ветку `ai-beatmaker`. Если открыт ноутбук из другой ветки или старая копия Colab, новые изменения там не появятся.

## Цель обучения

Цель — до 200 000 обучающих примеров: 10 000 синтетических композиций + до 190 000 MIDI из PDMX (Public Domain/CC0, без конфликта лицензии и при наличии файла). Фактическое число зависит от пригодных файлов. Если подходящих внешних файлов меньше 190 000, сборка с `--allow-shortfall` сохранит все пригодные файлы без подмены дубликатами.

**Всё хранится на Google Drive:** загрузка PDMX продолжает `.part`-файлы, распаковка пропускает уже обработанные MIDI, сборка датасета продолжает `.inprogress`, а модель сохраняет атомарный checkpoint каждые 25 шагов и после каждой эпохи. Если Colab отключится, открой эту тетрадь, выполни ячейки сверху вниз и продолжи обучение с `--resume`. Клон проекта теперь обновляется при каждом запуске.


**Восстановление после предыдущего запуска:** ячейка сборки сначала переносит уже скачанные PDMX.csv, mid.tar.gz, manifest и MIDI из `/content/honeydew/data/ai/external/pdmx` на Google Drive, затем продолжает распаковку. Датасет создаётся с `--allow-shortfall`: если 190 000 подходящих внешних MIDI нет, он честно соберёт все доступные, не создавая фиктивные дубликаты.


In [ ]:
!nvidia-smi || true
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


In [ ]:
# Всегда обновляем уже существующий клон: иначе Colab продолжит использовать старые скрипты.
!if [ ! -d /content/honeydew/.git ]; then git clone --depth 1 --branch ai-beatmaker https://github.com/djdhsvsbdb-design/honeydew.git /content/honeydew; else cd /content/honeydew && git fetch origin ai-beatmaker && git checkout ai-beatmaker && git reset --hard origin/ai-beatmaker; fi
%cd /content/honeydew
!python -m pip install -r tools/ai/requirements.txt


In [ ]:
from google.colab import drive
from pathlib import Path
import json
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Honeydew')
ROOT.mkdir(parents=True, exist_ok=True)
PDMX_DIR = ROOT / 'pdmx'
DATA_DIR = ROOT / 'data'
CHECKPOINT_DIR = ROOT / 'checkpoints'
for folder in (PDMX_DIR, DATA_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CHECKPOINT_DIR / 'honeydew-music.pt'
DATASET = DATA_DIR / 'honeydew-200k.jsonl'
STATUS = ROOT / 'pipeline-status.json'
def mark_stage(stage, details=None):
    payload = {'stage': stage, 'details': details or {}}
    STATUS.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Pipeline status saved:', payload)
print('Drive storage ready:', ROOT)
print('Checkpoint:', CHECKPOINT)
print('Dataset:', DATASET)


In [ ]:
# Переносим уже скачанный PDMX из временной папки Colab на Google Drive.
# Это предотвращает повторную загрузку архивов и сохраняет распакованные MIDI.
from pathlib import Path
import shutil, json, subprocess, sys

LOCAL_PDMX = Path('/content/honeydew/data/ai/external/pdmx')
PDMX_DIR = ROOT / 'pdmx'
DATA_DIR = ROOT / 'data'
CHECKPOINT_DIR = ROOT / 'checkpoints'
for folder in (PDMX_DIR, DATA_DIR, CHECKPOINT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

local_manifest = LOCAL_PDMX / 'external-manifest.jsonl'
drive_manifest = PDMX_DIR / 'external-manifest.jsonl'

if LOCAL_PDMX.exists() and (LOCAL_PDMX / 'PDMX.csv').exists():
    print('Найдена локальная загрузка PDMX:', LOCAL_PDMX, flush=True)
    # Копируем только если Drive ещё не содержит такой же архив.
    for filename in ('PDMX.csv', 'mid.tar.gz'):
        source = LOCAL_PDMX / filename
        target = PDMX_DIR / filename
        if source.exists() and not target.exists():
            print('Копирую', filename, 'на Google Drive...', flush=True)
            shutil.copy2(source, target)
    local_midi = LOCAL_PDMX / 'midi'
    drive_midi = PDMX_DIR / 'midi'
    if local_midi.exists():
        print('Копирую распакованные MIDI на Google Drive...', flush=True)
        shutil.copytree(local_midi, drive_midi, dirs_exist_ok=True)
    # Переписываем абсолютные пути в manifest на постоянные пути Google Drive.
    if local_manifest.exists():
        rewritten = []
        for line in local_manifest.open(encoding='utf-8'):
            try:
                row = json.loads(line)
                old_path = Path(row.get('path', ''))
                try:
                    relative = old_path.resolve().relative_to(LOCAL_PDMX.resolve())
                    row['path'] = str((PDMX_DIR / relative).resolve())
                except (ValueError, OSError):
                    pass
                rewritten.append(json.dumps(row, ensure_ascii=False))
            except (json.JSONDecodeError, TypeError):
                continue
        tmp = drive_manifest.with_suffix('.jsonl.tmp')
        tmp.write_text('\n'.join(rewritten) + ('\n' if rewritten else ''), encoding='utf-8')
        tmp.replace(drive_manifest)
        print('Перенесено записей manifest:', len(rewritten), flush=True)

# Повторный запуск загрузчика теперь продолжит распаковку с Google Drive.
# Уже записанные MIDI будут пропущены.
mark_stage('pdmx_resume_started', {'pdmx_dir': str(PDMX_DIR)})
subprocess.run([
    sys.executable, 'tools/ai/download_pdmx.py',
    '--out-dir', str(PDMX_DIR), '--limit', '190000'
], check=True)

# Собираем все доступные и прошедшие проверку лицензий примеры.
# --allow-shortfall не подменяет отсутствующие файлы дубликатами:
# он публикует честный датасет меньшего размера, если PDMX не даст 190k MIDI.
mark_stage('dataset_build_started', {'dataset': str(DATASET)})
subprocess.run([
    sys.executable, 'tools/ai/build_dataset.py',
    '--synthetic-songs', '10000',
    '--external-limit', '190000',
    '--external-manifest', str(PDMX_DIR / 'external-manifest.jsonl'),
    '--out', str(DATASET),
    '--allow-shortfall'
], check=True)

with DATASET.open(encoding='utf-8') as handle:
    dataset_rows = sum(1 for line in handle if line.strip())
mark_stage('dataset_ready', {'dataset': str(DATASET), 'lines': dataset_rows})
print('ГОТОВО. Строк в датасете:', dataset_rows, flush=True)
print('Файл датасета:', DATASET, flush=True)


In [ ]:
# Checkpoint сохраняется каждые 25 шагов и в конце каждой эпохи прямо на Google Drive.
# Повторный запуск с --resume продолжает с последнего checkpoint.
import subprocess, sys, torch
if not DATASET.is_file():
    raise FileNotFoundError(f'Датасет не найден: {DATASET}. Сначала успешно выполни ячейку сборки.')
print('Dataset size:', DATASET.stat().st_size, 'bytes')
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('ВНИМАНИЕ: GPU не подключён. Для ускорения выбери Runtime → Change runtime type → GPU, затем снова выполни ячейки сверху вниз.')
mark_stage('training_started', {'dataset': str(DATASET), 'checkpoint': str(CHECKPOINT)})
subprocess.run([sys.executable, 'tools/ai/train_music.py', '--data', str(DATASET), '--epochs', '20', '--batch-size', '8', '--block', '256', '--val-fraction', '0.1', '--save-every-steps', '25', '--out', str(CHECKPOINT), '--resume'], check=True)
mark_stage('training_finished', {'checkpoint': str(CHECKPOINT)})


In [ ]:
!python tools/ai/generate_music.py --checkpoint /content/drive/MyDrive/Honeydew/checkpoints/honeydew-music.best.pt --style trap --bars 8 --bpm 145 --out /content/drive/MyDrive/Honeydew/generated/honeydew-generated


In [ ]:
from google.colab import files
from pathlib import Path

for filename in [
    '/content/drive/MyDrive/Honeydew/checkpoints/honeydew-music.pt',
    '/content/drive/MyDrive/Honeydew/checkpoints/honeydew-music.best.pt',
    '/content/drive/MyDrive/Honeydew/generated/honeydew-generated.mid',
    '/content/drive/MyDrive/Honeydew/generated/honeydew-generated.json',
]:
    if Path(filename).exists():
        files.download(filename)
    else:
        print(f'Пока нет файла: {filename}')
